# MTA Service Reliability Analysis
## 02: Weather Correlation Analysis

---
### Imports

In [2]:
import pandas as pd
from pandas.tseries.holiday import USFederalHolidayCalendar
import numpy as np

import matplotlib.pyplot as plt
import plotly.express as px
import plotly.figure_factory as ff
import plotly.graph_objects as go
import statsmodels

import requests

import os

pd.set_option("display.max_columns", None)

---
### Load Data

In [3]:
# Create a dataset of US Federal Holidays
cal = USFederalHolidayCalendar()

holidays = cal.holidays(
    start="2020-01-01",
    end="2026-12-31"
)

holiday_df = holidays.to_frame(index=False, name="date")
holiday_df["is_holiday"] = 1

In [4]:
holiday_df.head()

,date,is_holiday
0,2020-01-01,1
1,2020-01-20,1
2,2020-02-17,1
3,2020-05-25,1
4,2020-07-03,1


In [5]:
holiday_df.tail()

,date,is_holiday
71,2026-09-07,1
72,2026-10-12,1
73,2026-11-11,1
74,2026-11-26,1
75,2026-12-25,1


In [6]:
otp_df = pd.read_csv("../data/processed/otp_clean.csv", parse_dates=["Month"])
delays_df = pd.read_csv("../data/processed/delays_clean.csv", parse_dates=["Month"])

In [7]:
otp_df.head()

,Month,division,Line,Day Type,num_on_time_trips,num_sched_trips,terminal_on_time_performance
0,2015-01-01,A DIVISION,1,1,6874,9017,0.762338
1,2015-01-01,A DIVISION,2,1,2920,6175,0.472874
2,2015-01-01,A DIVISION,3,1,4004,5834,0.686322
3,2015-01-01,A DIVISION,4,1,3692,7623,0.484324
4,2015-01-01,A DIVISION,5,1,3203,6491,0.493452


In [8]:
delays_df.head()

,Month,Line,Day Type,Crew Availability,External Factors,Infrastructure & Equipment,Operating Conditions,Planned ROW Work,Police & Medical
0,2020-01-01,1,1,3,2,43,104,35,75
1,2020-01-01,1,2,7,0,13,16,6,22
2,2020-01-01,2,1,8,5,41,80,37,77
3,2020-01-01,2,2,4,1,21,82,12,43
4,2020-01-01,3,1,2,1,24,55,15,32


In [9]:
# # NYC weather data via NOAA API
# url = "https://www.ncei.noaa.gov/access/services/data/v1"

# params = {
#     "dataset": "daily-summaries",
#     "stations": "USW00094728",  # NYC Central Park
#     "startDate": "2020-01-01",
#     "endDate": "2026-01-01",
#     "dataTypes": "TAVG,PRCP,SNOW",
#     "units": "metric",
#     "format": "json"
# }

# response = requests.get(url, params=params)
# weather = pd.DataFrame(response.json())

# weather["DATE"] = pd.to_datetime(weather["DATE"])
# weather = weather.rename(columns={
#     "DATE": "Month",
#     "TAVG": "avg_temp",
#     "PRCP": "precipitation",
#     "SNOW": "snow"
# })

# weather.head()

In [10]:
# # Save to CSV
# weather.to_csv("../data/processed/weather.csv", index=False)

In [11]:
weather = pd.read_csv("../data/processed/weather.csv", parse_dates=["Month"])
weather.head()

,Month,STATION,snow,precipitation
0,2020-01-01,USW00094728,0.0,0.0
1,2020-01-02,USW00094728,0.0,0.0
2,2020-01-03,USW00094728,0.0,3.8
3,2020-01-04,USW00094728,0.0,6.9
4,2020-01-05,USW00094728,0.0,0.0


---
### Merge Data

Merge on-terminal performance dataset with delays.

In [12]:
# Make a copy of on-time performance data for analysis
merged_otp_delays = otp_df.copy()

In [13]:
merged_otp_delays = merged_otp_delays.merge(
    delays_df,
    on=["Month", "Line", "Day Type"],
    how="left"
)

In [14]:
merged_otp_delays.head()

,Month,division,Line,Day Type,num_on_time_trips,num_sched_trips,terminal_on_time_performance,Crew Availability,External Factors,Infrastructure & Equipment,Operating Conditions,Planned ROW Work,Police & Medical
0,2015-01-01,A DIVISION,1,1,6874,9017,0.762338,NaN,NaN,NaN,NaN,NaN,NaN
1,2015-01-01,A DIVISION,2,1,2920,6175,0.472874,NaN,NaN,NaN,NaN,NaN,NaN
2,2015-01-01,A DIVISION,3,1,4004,5834,0.686322,NaN,NaN,NaN,NaN,NaN,NaN
3,2015-01-01,A DIVISION,4,1,3692,7623,0.484324,NaN,NaN,NaN,NaN,NaN,NaN
4,2015-01-01,A DIVISION,5,1,3203,6491,0.493452,NaN,NaN,NaN,NaN,NaN,NaN


In [15]:
merged_otp_delays.tail()

,Month,division,Line,Day Type,num_on_time_trips,num_sched_trips,terminal_on_time_performance,Crew Availability,External Factors,Infrastructure & Equipment,Operating Conditions,Planned ROW Work,Police & Medical
4997,2026-04-01,B DIVISION,R,2,1554,1724,0.901392,8.0,0.0,5.0,4.0,4.0,18.0
4998,2026-04-01,B DIVISION,S Fkln,1,5234,5236,0.999618,0.0,0.0,0.0,0.0,1.0,0.0
4999,2026-04-01,B DIVISION,S Fkln,2,1904,1904,1.000000,NaN,NaN,NaN,NaN,NaN,NaN
5000,2026-04-01,B DIVISION,S Rock,1,3286,3522,0.932993,3.0,21.0,5.0,4.0,15.0,3.0
5001,2026-04-01,B DIVISION,S Rock,2,305,312,0.977564,0.0,0.0,0.0,1.0,0.0,0.0


Filter rows to years 2020 - 2026.

In [16]:
merged_otp_delays = merged_otp_delays[merged_otp_delays["Month"] >= "2020-01-01"].reset_index(drop=True)

In [17]:
merged_otp_delays.head()

,Month,division,Line,Day Type,num_on_time_trips,num_sched_trips,terminal_on_time_performance,Crew Availability,External Factors,Infrastructure & Equipment,Operating Conditions,Planned ROW Work,Police & Medical
0,2020-01-01,A DIVISION,1,1,9035,10554,0.856074,3.0,2.0,43.0,104.0,35.0,75.0
1,2020-01-01,A DIVISION,1,2,2866,3050,0.939672,7.0,0.0,13.0,16.0,6.0,22.0
2,2020-01-01,A DIVISION,2,1,6058,7330,0.826467,8.0,5.0,41.0,80.0,37.0,77.0
3,2020-01-01,A DIVISION,2,2,1674,2340,0.715385,4.0,1.0,21.0,82.0,12.0,43.0
4,2020-01-01,A DIVISION,3,1,5732,6603,0.868090,2.0,1.0,24.0,55.0,15.0,32.0


In [18]:
len(merged_otp_delays)

3406

I used a left join to preserve the primary dataset (on-time performance),
then filtered the dataset to the time range where both datasets overlap (2020 onward).

In [19]:
# Check whether some of the values are NaN
merged_otp_delays.info()

<class 'pandas.DataFrame'>
RangeIndex: 3406 entries, 0 to 3405
Data columns (total 13 columns):
 #   Column                        Non-Null Count  Dtype         
---  ------                        --------------  -----         
 0   Month                         3406 non-null   datetime64[us]
 1   division                      3406 non-null   str           
 2   Line                          3406 non-null   str           
 3   Day Type                      3406 non-null   int64         
 4   num_on_time_trips             3406 non-null   int64         
 5   num_sched_trips               3406 non-null   int64         
 6   terminal_on_time_performance  3406 non-null   float64       
 7   Crew Availability             2958 non-null   float64       
 8   External Factors              2958 non-null   float64       
 9   Infrastructure & Equipment    2958 non-null   float64       
 10  Operating Conditions          2958 non-null   float64       
 11  Planned ROW Work              2958 non-nu

There are ~448 rows with null values, which means that they have no delay data. Replace such cells with 0 to keep data unbiased.

In [20]:
delay_cols = [
    "Crew Availability",
    "External Factors",
    "Infrastructure & Equipment",
    "Operating Conditions",
    "Planned ROW Work",
    "Police & Medical"
]

merged_otp_delays[delay_cols] = merged_otp_delays[delay_cols].fillna(0)

In [21]:
merged_otp_delays.info()

<class 'pandas.DataFrame'>
RangeIndex: 3406 entries, 0 to 3405
Data columns (total 13 columns):
 #   Column                        Non-Null Count  Dtype         
---  ------                        --------------  -----         
 0   Month                         3406 non-null   datetime64[us]
 1   division                      3406 non-null   str           
 2   Line                          3406 non-null   str           
 3   Day Type                      3406 non-null   int64         
 4   num_on_time_trips             3406 non-null   int64         
 5   num_sched_trips               3406 non-null   int64         
 6   terminal_on_time_performance  3406 non-null   float64       
 7   Crew Availability             3406 non-null   float64       
 8   External Factors              3406 non-null   float64       
 9   Infrastructure & Equipment    3406 non-null   float64       
 10  Operating Conditions          3406 non-null   float64       
 11  Planned ROW Work              3406 non-nu

Now all the values are non-null.

---
### Analysis

In [22]:
merged_otp_delays["op_delay_rate"] = merged_otp_delays["Operating Conditions"] / merged_otp_delays["num_sched_trips"]

fig = px.scatter(
    merged_otp_delays,
    x="op_delay_rate",
    y="terminal_on_time_performance",
    custom_data=["Line", "Month"],
    opacity=0.4,
    trendline="ols",
    title="Impact of Operating Condition Delays on On-Time Performance",
    labels={
        "op_delay_rate": "Share of Trips Affected by Operational Delays",
        "terminal_on_time_performance": "% of Trains Arriving On Time"
    }
)

fig.update_yaxes(range=[0.5, 1], tickformat=".0%")
fig.update_traces(
    hovertemplate=
    "Line: %{customdata[0]}<br>" +
    "Month: %{customdata[1]|%Y-%m}<br>" +
    "Operational Delay Rate: %{x:.2%}<br>" +
    "On-Time Performance: %{y:.1%}<extra></extra>"
)

fig.update_layout(template="plotly_white")
fig.show()

**Insight:**

There is a clear negative relationship between operational delays and on-time performance. 
As the proportion of trips affected by operational issues increases, the percentage of trains arriving on time decreases.

Most observations are concentrated at low delay rates, indicating that the system typically operates with relatively few disruptions. 
However, even small increases in delay rates are associated with noticeable declines in performance.

Additionally, variability in on-time performance increases at higher delay rates, suggesting that the system becomes less predictable during periods of disruption.

In [23]:
merged_otp_delays["delay_per_1000_trips"] = (
    merged_otp_delays["op_delay_rate"] * 1000
)

fig = px.scatter(
    merged_otp_delays,
    x="delay_per_1000_trips",
    y="terminal_on_time_performance",
    trendline="lowess",
    opacity=0.3,
    title="Higher Operational Delays Are Associated with Lower On-Time Performance",
    labels={
        "delay_per_1000_trips": "Delays per 1,000 Trips",
        "terminal_on_time_performance": "On-Time Performance (%)"
    }
)

fig.update_traces(marker=dict(size=4))

fig.update_yaxes(tickformat=".0%")

fig.update_layout(
    template="plotly_white",
    title_x=0.5,
    width=900,
    height=500
)

fig.show()

**Insight:**

There is a strong negative relationship between operational delays and on-time performance. 
As delays per 1,000 trips increase, the percentage of trains arriving on time decreases significantly.

Even small increases in delays lead to noticeable declines in performance, indicating that the system is highly sensitive to operational disruptions. 

Additionally, variability in performance increases at higher delay levels, suggesting that the system becomes less predictable under stress.

This shows that minimizing operational delays is critical not only for improving average performance, but also for maintaining system reliability and consistency.

In [24]:
monthly_otp = (
    merged_otp_delays
    .groupby("Month")["terminal_on_time_performance"]
    .mean()
    .reset_index()
)

monthly_delays = merged_otp_delays.groupby("Month")["op_delay_rate"].mean().reset_index()

monthly = monthly_otp.merge(monthly_delays, on="Month")

fig = go.Figure()

# On-time performance
fig.add_trace(go.Scatter(
    x=monthly["Month"],
    y=monthly["terminal_on_time_performance"],
    name="On-Time Performance",
    yaxis="y1"
))

# Delays
fig.add_trace(go.Scatter(
    x=monthly["Month"],
    y=monthly["op_delay_rate"],
    name="Total Delays",
    yaxis="y2"
))

fig.update_layout(
    title="On-Time Performance vs Total Delays Over Time",
    xaxis=dict(title="Month"),
    yaxis=dict(title="On-Time Performance (%)", tickformat=".0%"),
    yaxis2=dict(
        title="Operational Delay Rate (%)",
        overlaying="y",
        side="right"
    ),
    template="plotly_white"
)

fig.show()

**Insight:**

Here, we observe a consistent inverse relationship between operational delays and on-time performance over time. Notably, from 2021 to 2022, increasing delays coincide with a sustained drop in reliability, while stabilization in delays after 2023 aligns with gradual recovery.

In [25]:
delay_cols = [
    "Crew Availability",
    "External Factors",
    "Infrastructure & Equipment",
    "Operating Conditions",
    "Planned ROW Work",
    "Police & Medical"
]

delays_by_category = (
    delays_df[delay_cols]
    .sum()
    .sort_values()
    .reset_index()
)

delays_by_category.columns = ["Reporting Category", "Incidents"]

delays_by_category["share"] = (
    delays_by_category["Incidents"] /
    delays_by_category["Incidents"].sum()
)

delays_by_category = delays_by_category.sort_values("share", ascending=False)

In [26]:
fig = px.pie(
    delays_by_category,
    values="share",
    names="Reporting Category",
    hole=0.4,
    title="Share of Total Delays by Category"
)

fig.update_traces(textinfo="percent+label", textposition="outside")
fig.update_layout(template="plotly_white")

fig.show()

**Insight:**

Roughly 70% of subway delays are driven by three categories:Police & Medical, Operating Conditions, and Infrastructure, highlighting that while some delays are external, a significant portion comes from operational inefficiencies and system limitations.

In [27]:
label_map = {
    "terminal_on_time_performance": "On-Time Performance",
    "op_delay_rate": "Delay Rate",
    "Crew Availability": "Crew Availability",
    "External Factors": "External Factors",
    "Infrastructure & Equipment": "Infrastructure & Equipment",
    "Operating Conditions": "Operating Conditions",
    "Planned ROW Work": "Planned Work",
    "Police & Medical": "Police & Medical"
}

cols = [
    "terminal_on_time_performance",
    "op_delay_rate",
    "Crew Availability",
    "External Factors",
    "Infrastructure & Equipment",
    "Operating Conditions",
    "Planned ROW Work",
    "Police & Medical"
]

corr = merged_otp_delays[cols].corr().round(2)
corr_pretty = corr.rename(index=label_map, columns=label_map)

fig = px.imshow(
    corr_pretty,
    text_auto=True,
    color_continuous_scale="RdBu",
    aspect="auto"
)

fig.update_traces(texttemplate="%{z:.2f}")

fig.update_layout(
    title="Correlation Between Delays and On-Time Performance",
    width=900,
    height=700
)

fig.update_xaxes(tickangle=45)

fig.show()

**Insight:**

On-time performance is most strongly affected by operational and infrastructure-related delays, while external and staffing factors have relatively limited impact. Additionally, strong correlations between delay categories suggest cascading system failures rather than isolated incidents

In [28]:
line_perf = merged_otp_delays.groupby("Line")["terminal_on_time_performance"].mean().reset_index()

top_bottom = pd.concat([
    line_perf.nsmallest(5, "terminal_on_time_performance"),
    line_perf.nlargest(5, "terminal_on_time_performance")
])

top_bottom["group"] = ["Bottom"]*5 + ["Top"]*5

fig = px.bar(
    top_bottom.sort_values("terminal_on_time_performance"),
    x="terminal_on_time_performance",
    y="Line",
    color="group",
    orientation="h",
    title="Subway Line Reliability: Top vs Bottom Performers",
    labels={
        "terminal_on_time_performance": "On-Time Performance (%)",
        "Line": "Subway Line",
        "group": "Category"
    }
)

fig.update_layout(template="plotly_white")
fig.update_xaxes(tickformat=".0%")

fig.update_traces(texttemplate='%{x:.1%}', textposition='outside')

fig.show()

**Insight:**

There is a stark contrast in on-time performance across subway lines. Shorter shuttle and limited-service lines (e.g., S 42nd, S Fkln, GS) achieve near-perfect reliability (~99–100%), while major, longer lines (e.g., 2, B, N, C) perform significantly worse, with on-time rates around 72–75%.

This ~25–30 percentage point gap suggests that route complexity, length, and exposure to operational disruptions are key drivers of reliability. Simpler, more contained lines are inherently more resilient, whereas heavily interconnected lines are more vulnerable to cascading delays.

---
### Add Weather and US Holidays 

Before merging, make sure that all datasets align on Month or date.

In [29]:
# Main dataset
merged_otp_delays["Month"] = pd.to_datetime(merged_otp_delays["Month"])

# Holidays
holiday_df["date"] = pd.to_datetime(holiday_df["date"])
holiday_df["Month"] = holiday_df["date"].dt.to_period("M").dt.to_timestamp()

Aggregate holidays.

In [30]:
holiday_monthly = (
    holiday_df
    .groupby("Month")["is_holiday"]
    .sum()
    .reset_index(name="num_holidays")
)
holiday_monthly.head()

,Month,num_holidays
0,2020-01-01,2
1,2020-02-01,1
2,2020-05-01,1
3,2020-07-01,1
4,2020-09-01,1


Aggregate weather because we have multiple stations per month.

In [31]:
weather_monthly = (
    weather
    .groupby("Month")
    .agg({
        "snow": "sum",
        "precipitation": "sum"
    })
    .reset_index()
)

In [32]:
weather_monthly.head()

,Month,snow,precipitation
0,2020-01-01,0.0,0.0
1,2020-01-02,0.0,0.0
2,2020-01-03,0.0,3.8
3,2020-01-04,0.0,6.9
4,2020-01-05,0.0,0.0


Merge all 3 dataframes.

In [33]:
final_df = merged_otp_delays.merge(weather_monthly, on="Month", how="left")
final_df = final_df.merge(holiday_monthly, on="Month", how="left")
final_df["num_holidays"] = final_df["num_holidays"].fillna(0)

In [34]:
final_df.head()

,Month,division,Line,Day Type,num_on_time_trips,num_sched_trips,terminal_on_time_performance,Crew Availability,External Factors,Infrastructure & Equipment,Operating Conditions,Planned ROW Work,Police & Medical,op_delay_rate,delay_per_1000_trips,snow,precipitation,num_holidays
0,2020-01-01,A DIVISION,1,1,9035,10554,0.856074,3.0,2.0,43.0,104.0,35.0,75.0,0.009854,9.854084,0.0,0.0,2.0
1,2020-01-01,A DIVISION,1,2,2866,3050,0.939672,7.0,0.0,13.0,16.0,6.0,22.0,0.005246,5.245902,0.0,0.0,2.0
2,2020-01-01,A DIVISION,2,1,6058,7330,0.826467,8.0,5.0,41.0,80.0,37.0,77.0,0.010914,10.914052,0.0,0.0,2.0
3,2020-01-01,A DIVISION,2,2,1674,2340,0.715385,4.0,1.0,21.0,82.0,12.0,43.0,0.035043,35.042735,0.0,0.0,2.0
4,2020-01-01,A DIVISION,3,1,5732,6603,0.868090,2.0,1.0,24.0,55.0,15.0,32.0,0.008330,8.329547,0.0,0.0,2.0


In [35]:
# monthly = (
#     final_df
#     .groupby("Month")
#     .agg({
#         "terminal_on_time_performance": "mean",
#         "snow": "mean"
#     })
#     .reset_index()
# )

In [40]:
final_df.columns

Index(['Month', 'division', 'Line', 'Day Type', 'num_on_time_trips',
       'num_sched_trips', 'terminal_on_time_performance', 'Crew Availability',
       'External Factors', 'Infrastructure & Equipment',
       'Operating Conditions', 'Planned ROW Work', 'Police & Medical',
       'op_delay_rate', 'delay_per_1000_trips', 'snow', 'precipitation',
       'num_holidays'],
      dtype='str')

In [41]:
delay_cols = [
    'Crew Availability',
    'External Factors',
    'Infrastructure & Equipment',
    'Operating Conditions',
    'Planned ROW Work',
    'Police & Medical'
]

long_df = final_df.melt(
    id_vars=["Line"],
    value_vars=delay_cols,
    var_name="Reporting Category",
    value_name="Value"
)

In [42]:
long_df.head()

,Line,Reporting Category,Value
0,1,Crew Availability,3.0
1,1,Crew Availability,7.0
2,2,Crew Availability,8.0
3,2,Crew Availability,4.0
4,3,Crew Availability,2.0


In [45]:
line_category_counts = (
    long_df
    .groupby(["Line", "Reporting Category"])["Value"]
    .sum()
    .reset_index()
)

In [46]:
line_category = (
    long_df
    .groupby(["Line", "Reporting Category"])["Value"]
    .sum()
    .reset_index()
)

In [47]:
line_category["pct"] = (
    line_category.groupby("Line")["Value"]
    .transform(lambda x: x / x.sum())
)

In [51]:
fig = px.bar(
    line_category,
    x="Line",
    y="pct",
    color="Reporting Category",
    title="Delay Distribution per Subway Line",
    labels={"pct": "Proportion of Delays"},
)

fig.update_layout(barmode="stack")
fig.show()

---
### Analysis with Weather and Holidays

In [109]:
cols = [
    "terminal_on_time_performance",
    "op_delay_rate",
    "snow",
    "precipitation",
    "num_holidays"
]

label_map = {
    "terminal_on_time_performance": "On-Time Performance (%)",
    "op_delay_rate": "Delay Rate",
    "snow": "Snow",
    "precipitation": "Rain",
    "num_holidays": "Holidays"
}

corr_pretty = final_df[cols].corr().round(2)
corr_pretty = corr_pretty.rename(index=label_map, columns=label_map)

fig = px.imshow(
    corr_pretty,
    text_auto=True,
    color_continuous_scale="RdBu",
    aspect="auto"
)

fig.update_layout(
    title="Impact of Weather & Holidays on Subway Performance",
    width=1100,
    height=900
)

fig.update_traces(texttemplate="%{z:.2f}", textfont_size=14)
fig.update_xaxes(tickfont=dict(size=12))
fig.update_yaxes(tickfont=dict(size=12))

fig.show()

**Insight:**

External factors such as weather and holidays show minimal correlation with subway performance at the monthly level. This suggests that system reliability is primarily driven by internal operational factors rather than environmental conditions.

In [110]:
final_df["heavy_snow"] = final_df["snow"] >= 50

In [111]:
final_df.head()

,Month,division,Line,Day Type,num_on_time_trips,num_sched_trips,terminal_on_time_performance,Crew Availability,External Factors,Infrastructure & Equipment,Operating Conditions,Planned ROW Work,Police & Medical,op_delay_rate,delay_per_1000_trips,snow,precipitation,num_holidays,heavy_snow
0,2020-01-01,A DIVISION,1,1,9035,10554,0.856074,3.0,2.0,43.0,104.0,35.0,75.0,0.009854,9.854084,0.0,0.0,2.0,False
1,2020-01-01,A DIVISION,1,2,2866,3050,0.939672,7.0,0.0,13.0,16.0,6.0,22.0,0.005246,5.245902,0.0,0.0,2.0,False
2,2020-01-01,A DIVISION,2,1,6058,7330,0.826467,8.0,5.0,41.0,80.0,37.0,77.0,0.010914,10.914052,0.0,0.0,2.0,False
3,2020-01-01,A DIVISION,2,2,1674,2340,0.715385,4.0,1.0,21.0,82.0,12.0,43.0,0.035043,35.042735,0.0,0.0,2.0,False
4,2020-01-01,A DIVISION,3,1,5732,6603,0.868090,2.0,1.0,24.0,55.0,15.0,32.0,0.008330,8.329547,0.0,0.0,2.0,False


In [112]:
# Snow
fig = px.box(
    final_df,
    x="heavy_snow",
    y="terminal_on_time_performance",
    labels={
        "heavy_snow": "Heavy Snow (>= 50mm)",
        "terminal_on_time_performance": "On-Time Performance (%)"
    },
    title="Impact of Heavy Snow on Subway Performance"
)
fig.update_yaxes(tickformat=".0%")
fig.show()

**Insight:**

Contrary to expectations, heavy snowfall does not significantly degrade subway on-time performance. Instead, performance becomes more consistent, with fewer extreme delays. This suggests that the system shifts into a controlled operating mode during severe weather, prioritizing reliability over throughput.

In [113]:
# Rain
final_df["rain_level"] = pd.cut(
    final_df["precipitation"],
    bins=[-1, 0, 10, 50, 200],
    labels=["No Rain", "Light Rain", "Moderate Rain", "Heavy Rain"]
)

fig = px.box(
    final_df,
    x="rain_level",
    y="terminal_on_time_performance",
    title="Impact of Rain Intensity on Performance",
    labels={
        "rain_level": "Rain Intensity",
        "terminal_on_time_performance": "On-Time Performance (%)"
    }
)

fig.update_yaxes(tickformat=".0%")
fig.show()

In [114]:
final_df["rain_level"].value_counts()

rain_level
No Rain          2017
Light Rain        760
Moderate Rain     449
Heavy Rain         45
Name: count, dtype: int64

**Insight:**

Unlike heavy snow, which triggers operational adjustments that stabilize performance, increasing rainfall introduces mild inefficiencies without prompting system-wide adaptation. As a result, on-time performance slightly declines and becomes more variable under heavier rain, though the overall system remains resilient.

In [115]:
monthly = (
    final_df
    .groupby("Month")
    .agg({
        "terminal_on_time_performance": "mean",
        "snow": "median" 
    })
    .reset_index()
)

monthly["snow_log"] = np.log1p(monthly["snow"])
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=monthly["Month"],
    y=monthly["terminal_on_time_performance"],
    name="On-Time Performance",
    yaxis="y1"
))

fig.add_trace(go.Scatter(
    x=monthly["Month"],
    y=monthly["snow_log"],
    name="Snowfall",
    yaxis="y2"
))

fig.update_layout(
    title="Monthly Performance vs Snowfall",
    yaxis=dict(title="On-Time Performance (%)", tickformat=".0%"),
    yaxis2=dict(title="Snow", overlaying="y", side="right"),
    template="plotly_white"
)

fig.show()

In [116]:
final_df.groupby(final_df["snow"] > 0)["terminal_on_time_performance"].mean()

snow
False    0.838028
True     0.844986
Name: terminal_on_time_performance, dtype: float64

**Insight:**

Snowfall, including extreme outlier events, does not appear to be a primary driver of subway performance variability.

In [117]:
monthly_holidays = (
    final_df
    .groupby("Month")["num_holidays"]
    .mean()
    .reset_index()
)

In [118]:
final_df.head()


,Month,division,Line,Day Type,num_on_time_trips,num_sched_trips,terminal_on_time_performance,Crew Availability,External Factors,Infrastructure & Equipment,Operating Conditions,Planned ROW Work,Police & Medical,op_delay_rate,delay_per_1000_trips,snow,precipitation,num_holidays,heavy_snow,rain_level
0,2020-01-01,A DIVISION,1,1,9035,10554,0.856074,3.0,2.0,43.0,104.0,35.0,75.0,0.009854,9.854084,0.0,0.0,2.0,False,No Rain
1,2020-01-01,A DIVISION,1,2,2866,3050,0.939672,7.0,0.0,13.0,16.0,6.0,22.0,0.005246,5.245902,0.0,0.0,2.0,False,No Rain
2,2020-01-01,A DIVISION,2,1,6058,7330,0.826467,8.0,5.0,41.0,80.0,37.0,77.0,0.010914,10.914052,0.0,0.0,2.0,False,No Rain
3,2020-01-01,A DIVISION,2,2,1674,2340,0.715385,4.0,1.0,21.0,82.0,12.0,43.0,0.035043,35.042735,0.0,0.0,2.0,False,No Rain
4,2020-01-01,A DIVISION,3,1,5732,6603,0.868090,2.0,1.0,24.0,55.0,15.0,32.0,0.008330,8.329547,0.0,0.0,2.0,False,No Rain


In [119]:
px.box(
    final_df,
    x="num_holidays",
    y="terminal_on_time_performance",
    title="Performance Distribution by Number of Holidays",
    labels={
        "num_holidays": "Number of Holidays",
        "terminal_on_time_performance": "On-Time Performance (%)"
    }
)

In [123]:
# Create holiday vs non-holiday label
final_df["holiday_label"] = final_df["num_holidays"].apply(
    lambda x: "Holiday" if x > 0 else "Non-Holiday"
)

In [124]:
holiday_summary = (
    final_df
    .groupby("holiday_label")["terminal_on_time_performance"]
    .mean()
    .reset_index()
)

In [125]:
fig = px.box(
    final_df,
    x="holiday_label",
    y="terminal_on_time_performance",
    title="On-Time Performance: Holiday vs Non-Holiday"
)

fig.show()

**Insight:**

The distribution of on-time performance remains consistent across months with varying numbers of holidays. Median performance, spread, and variability show no meaningful differences between 0, 1, or 2 holiday months, indicating that holidays do not significantly impact system reliability.

In [52]:
corr = final_df[[
    "terminal_on_time_performance",
    "Crew Availability",
    "Infrastructure & Equipment",
    "Operating Conditions",
    "Police & Medical"
]].corr()

corr_pretty = corr.rename(index=label_map, columns=label_map)

fig = px.imshow(
    corr_pretty,
    text_auto=".2f",
    color_continuous_scale="RdBu",
    zmin=-1, zmax=1,
    aspect="auto"
)

fig.update_layout(
    title="Key Drivers of Subway Performance",
    title_x=0.5,
    font=dict(size=14),
)

fig.update_xaxes(side="top")

fig.show()

**Insight:**

The analysis revealed that performance degradation is not driven by external variability, but by tightly coupled internal failure modes. Factors like infrastructure issues, operating conditions, and police or medical incidents are both strongly correlated with performance drops and with each other, indicating cascading disruptions within the system.

In [59]:
impact = corr["terminal_on_time_performance"] \
    .drop("terminal_on_time_performance") \
    .sort_values()

impact.index = ["Crew", "Infrastructure", "Incidents", "Operations"]

fig = px.bar(
    impact,
    orientation="h",
    title="What Drives Subway Performance?",
    labels={"value": "Correlation", "index": "Factor"},
    color=impact.values,
    color_continuous_scale=["#8B0000", "#FF6B6B"], 
)

fig.update_layout(
    title_x=0.5,
    template="plotly_white",
    showlegend=False,
    coloraxis_showscale=False,
    font=dict(size=14),
    margin=dict(l=120, r=40, t=60, b=40)
)

fig.update_traces(
    text=impact.round(2),
    textposition="outside"
)

fig.add_vline(x=0, line_dash="dash", line_color="black")

fig.show()